# Setup Sagemaker

## Imports

In [1]:
import sagemaker
import boto3
from sagemaker.tuner import (
    IntegerParameter,
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner,
    WarmStartConfig,
    WarmStartTypes
)
from sagemaker.tensorflow import TensorFlow

client = boto3.client('sagemaker')
sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
role = sagemaker.get_execution_role()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Helper Functions

In [2]:
def get_small_estimator(hparams_dict, n_instances=1):
    estimator = TensorFlow(
        entry_point="ML.py",
        role=role,
        source_dir=source_uri,
        model_dir=model_uri,
        framework_version="2.13",
        py_version="py310",
        instance_type="ml.g5.xlarge",
        instance_count=n_instances,
        volume_size=20,
        output_path=output_uri,
        hyperparameters=hparams_dict
    )
    return estimator


def get_distributed_estimator(hparams_dict, n_instances=1):
    if hparams_dict["distributed"] == "tf":
        dist_config = {"multi_worker_mirrored_strategy": {"enabled": True}}
        env_vars = None
    elif hparams_dict["distributed"] == "horovod":
        dist_config = {"mpi": {"enabled": True}}
        env_vars = {"HOROVOD_GPU_OPERATIONS": "NCCL"}

    estimator = TensorFlow(
        entry_point="ML.py",
        role=role,
        source_dir=source_uri,
        model_dir=model_uri,
        framework_version="2.13",
        py_version="py310",
        instance_type="ml.g5.12xlarge",
        instance_count=n_instances,
        volume_size=20,
        output_path=output_uri,
        hyperparameters=hparams_dict,
        distribution=dist_config,
        environment=env_vars,
        checkpoint_s3_uri=checkpoint_uri
    )
    return estimator

# TODO: Add another function with file_mode as "pipe" or "fast_file"

# Upload data to S3

In [3]:
model_prefix = "my_first_model"
source_endpt = f"{model_prefix}/inputs/source"
inputs_endpt = f"{model_prefix}/inputs/datasets"

## Upload to source_uri

In [48]:
local_dir = "WeThePeople.tgz" # when you make this zip file, don't include the datasets folder. Seriously.
inputs = sagemaker_session.upload_data(path=local_dir, bucket=bucket, key_prefix=source_endpt)
print("input spec (in this case, just an S3 path): {}".format(inputs))

input spec (in this case, just an S3 path): s3://sagemaker-us-east-1-533267063987/my_first_model/inputs/source/WeThePeople.tgz


## Upload to train_data_uri

In [15]:
local_dir = "datasets"
inputs = sagemaker_session.upload_data(path=local_dir, bucket=bucket, key_prefix=inputs_endpt)
print("input spec (in this case, just an S3 path): {}".format(inputs))

input spec (in this case, just an S3 path): s3://sagemaker-us-east-1-533267063987/my_first_model/inputs/datasets


# Sagemaker Estimator

In [4]:
train_data_uri = f"s3://{bucket}/{inputs_endpt}/training data"
model_uri = f"s3://{bucket}/{model_prefix}/outputs/model"
output_uri = f"s3://{bucket}/{model_prefix}/outputs/output"
source_uri = f"s3://{bucket}/{source_endpt}/WeThePeople.tgz"
checkpoint_uri = f"s3://{bucket}/{model_prefix}/outputs/checkpoints"
channels = {
    "train": train_data_uri
}

## SINGLE TRAINING JOBS

### Setup

In [50]:
# Customize as needed
hparams = {
    "num_threads": 2,
    "batch_size_per_worker": 1,
    "n_vocab": 100000,
    "label_seq_length": 20,
    "series_seq_length": 40,
    "dropout_rate": 0.1,
    "num_heads": 2,
    "stack_height": 1,
    "d_values": 12,
    "d_keys": 12,
    "encoder_max_seq_len": 512,
    "epochs": 3,
    "learning_rate": 1e-3
}

### Run Model

#### Non-distributed

In [ ]:
estimator = get_small_estimator(hparams)
estimator.fit(inputs=channels)

#### Distributed

In [ ]:
hparams["distributed"] = "horovod"
estimator = get_distributed_estimator(hparams)
estimator.fit(inputs=channels)

## TUNING JOBS

### Setup Estimator

In [5]:
# Customize as needed
static_hparams = {
    "epochs": 4,
    "n_vocab": 100000,
    "label_seq_length": 20,
    "series_seq_length": 40
}

#### Distributed

In [6]:
static_hparams["distributed"] = "horovod"
estimator = get_distributed_estimator(static_hparams)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


#### Non-distributed

In [58]:
estimator = get_small_estimator(static_hparams)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


### Setup Tuner

In [8]:
hparam_ranges = {"encoder_max_seq_len": IntegerParameter(627, 900),
                 "batch_size_per_worker": IntegerParameter(3, 7),
                 "num_threads": IntegerParameter(17, 27),
                 "d_values": IntegerParameter(447, 675),
                 "d_keys": IntegerParameter(191, 300),
                 "num_heads": IntegerParameter(11, 18),
                 "stack_height": IntegerParameter(3, 6),
                 "dropout_rate": ContinuousParameter(0, 0.2),
                 "learning_rate": ContinuousParameter(1e-3, 6e-3)
                }

obj_metric_name = "MAPE"
obj_type = "Minimize"
metric_defs = [
    {
        "Name": "MSE",
        "Regex": "val_mse: ([0-9\\.]+)"
    },
    {
        "Name": "MAPE",
        "Regex": "val_mape: ([0-9\\.]+)"
    },
    {
        "Name": "MAE",
        "Regex": "val_mae: ([0-9\\.]+)"
    },
    {
        "Name": "TEST_MSE",
        "Regex": "test_mse: ([0-9\\.]+)"
    },
    {
        "Name": "TEST_MAPE",
        "Regex": "test_mape: ([0-9\\.]+)"
    },
    {
        "Name": "TEST_MAE",
        "Regex": "test_mae: ([0-9\\.]+)"
    },
]


# Setup warm start config
parent_names = {"TwelveXlargeTune-240404-2029"}
warm_start_config = WarmStartConfig(
    WarmStartTypes.IDENTICAL_DATA_AND_ALGORITHM, parents=parent_names
)

# Initialize tuner
tuner = HyperparameterTuner(
    base_tuning_job_name="TwelveXlargeTune",
    estimator=estimator,
    objective_metric_name=obj_metric_name,
    hyperparameter_ranges=hparam_ranges,
    metric_definitions=metric_defs,
    objective_type=obj_type,
    max_jobs=10,
    max_parallel_jobs=1,
    warm_start_config=warm_start_config
)


### Run Tuner

In [ ]:
tuner.fit(inputs=channels, include_cls_metadata=False)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


..................

NOTES

For info on successful runs so far, see the screenshot in your WeThePeople folder